# Properati Argentina: Predicting with Size, Location and Neighborhood


In [1]:
# Imports
import csv
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, spearmanr, pearsonr
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
# from statsmodels.stats.outliers_influence import variance_inflation_factor
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## File paths and reproducibility

The original notebook repairs `entrenamiento.csv` because some rows contain more than the expected 25 fields, usually because commas occur inside the description. The repair logic is retained, but now reports and validates the result.

In [2]:
RAW_FILE = Path("entrenamiento.csv")
PARSED_FILE = Path("entrenamiento_parsed.csv")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_COLUMNS = 25

In [ ]:
def reconstruct_csv(input_file, output_file, expected_columns=25):
    valid = repaired = rejected = 0
    with open(input_file, "r", encoding="utf-8", newline="") as infile, open(output_file, "w", encoding="utf-8", newline="") as outfile:
        reader, writer = csv.reader(infile), csv.writer(outfile)
        header = next(reader)
        writer.writerow(header)
        if len(header) != expected_columns:
            raise ValueError(f"Expected {expected_columns} columns, got {len(header)}")
        for row in reader:
            if len(row) == expected_columns:
                writer.writerow(row); valid += 1
            elif len(row) > expected_columns:
                parsed = row[:21] + [",".join(row[21:-3])] + row[-3:]
                if len(parsed) == expected_columns:
                    writer.writerow(parsed); repaired += 1
                else: rejected += 1
            else: rejected += 1
    return valid, repaired, rejected

if not RAW_FILE.exists():
    raise FileNotFoundError("Place entrenamiento.csv beside this notebook.")
valid, repaired, rejected = reconstruct_csv(RAW_FILE, PARSED_FILE)
print(f"Valid: {valid:,} | Repaired: {repaired:,} | Rejected: {rejected:,}")

In [ ]:
data = pd.read_csv(PARSED_FILE)
assert len(data.columns) == EXPECTED_COLUMNS
print("Shape:", data.shape)

# data.columns

##  Data-quality assessment

In [ ]:
# data.head()
columns1 = ['lat', 'lon','l1', 'l2', 'l3', 'rooms', 'bedrooms', 'bathrooms', 
           'surface_total', 'surface_covered', 'currency', 'property_type', 'operation_type','price']

df = data[columns1]

In [ ]:
def clean(df):
    # Create a copy to prevent SettingWithCopyWarning
    df_clean = df.copy()

    # Rename geographical columns & label neighborhood
    df_clean = df_clean.rename(columns={"lat": "lon", "lon": "lat"})
    df_clean = df_clean.rename(columns={"l1": "country", "l2": "province", "l3": "neighborhood"})

    # Subset Price data to only consider USD currency
    mask_USD = df_clean["currency"] == "USD"
    # Subset properati data: to only consider homes amongst the properties given 
    mask_home_props = df_clean["property_type"] == "Casa"
    # Subset location data: to only consider Argentina 
    mask_country = df_clean["country"] == "Argentina"

    # Median latitude and longitude by neighborhood
    df_clean["lat"] = df_clean["lat"].fillna(df_clean.groupby("neighborhood")["lat"].transform("median"))
    df_clean["lon"] = df_clean["lon"].fillna(df_clean.groupby("neighborhood")["lon"].transform("median"))
    
    # Fallback to province median
    df_clean["lat"] = df_clean["lat"].fillna(df_clean.groupby("province")["lat"].transform("median"))
    df_clean["lon"] = df_clean["lon"].fillna(df_clean.groupby("province")["lon"].transform("median"))
    
    # Fallback to country median
    df_clean["lat"] = df_clean["lat"].fillna(df_clean.groupby("country")["lat"].transform("median"))
    df_clean["lon"] = df_clean["lon"].fillna(df_clean.groupby("country")["lon"].transform("median"))
    
    # Specify column categories 
    num_cols = ["lat", "lon", "rooms", "bedrooms", "bathrooms", "surface_total", "surface_covered"]
    cat_cols = ["country", "province", "neighborhood", "currency", "property_type", "operation_type"]

     # Fill Numeric columns by → median
    num_cols = df_clean.select_dtypes(include="number").columns
    df_clean[num_cols] = SimpleImputer(strategy="median").fit_transform(df_clean[num_cols])
    
    # Fill Categorical columns by → most frequent value
    cat_cols = df_clean.select_dtypes(exclude="number").columns
    df_clean[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df_clean[cat_cols])

    # Subset data: Remove outliers for "surface_covered"
    low, high = df_clean["surface_covered"].quantile([0.1, 0.9])
    mask_area = df_clean["surface_covered"].between(low, high)
    df_clean = df_clean[mask_area & mask_USD & mask_home_props & mask_country]
    
    return df_clean

In [ ]:
df_clean = clean(df)
df_clean.head()

In [ ]:
columns2 = ['lat', 'lon', 'neighborhood', 'surface_covered', 'property_type', 'operation_type', 'price']

dataframe = df_clean[columns2]

# Chi-square test: property type vs operation type

**Null hypothesis (H₀):** There is no relationship between `property_type` and `operation_type` (they are independent).

**Alternative hypothesis (H₁):** There is a significant relationship between `property_type` and `operation_type`.

In [ ]:
# Contingency table
table = pd.crosstab(dataframe["property_type"], dataframe["operation_type"])

# Chi-square test
chi2, p , dof, expected = chi2_contingency(table)
print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

Since p-value = 1 > 0.05 → Fail to Reject H₀ ; hence no relationship between property_type and operation_type i.e. they are independent.

In [ ]:
# Effect size: Cramér's V
n = table.sum().sum()
phi2 = chi2 / n
r, k = table.shape
cramers_v = np.sqrt(phi2 / min(k-1, r-1))

print("Cramér's V:", cramers_v)

### Split Data

In [ ]:
target = "price"
features = ["surface_covered", "lat","lon","neighborhood"]
X_train=dataframe[features]
y_train=dataframe[target]

## Build Model

### Baseline

In [ ]:
y_mean = y_train.mean()
print("Mean home price:", y_mean.round(3))
y_pred_baseline = [y_mean]*len(y_train)
print("Baseline MAE:", mean_absolute_error(y_train, y_pred_baseline))

### Iterate

In [ ]:
# Identify column types
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_features = X_train.select_dtypes(include=["object", "category"]).columns

# Numeric preprocessing
num_transformer = make_pipeline(SimpleImputer(strategy="median"))

# Categorical preprocessing
cat_transformer = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[("num", num_transformer, num_features), ("cat", cat_transformer, cat_features)])

# Complete model pipeline
model = make_pipeline(preprocessor, Ridge())

# Fit
model.fit(X_train, y_train)

In [ ]:
# X_test = 

## Communicate Results

In [ ]:
def make_prediction(area, lat, lon, neighborhood):
    data = {"surface_covered":area, "lat":lat, "lon":lon, "neighborhood":neighborhood}
    
    df_clean = pd.DataFrame(data,index=[0])
    prediction = model.predict(df_clean).round(2)[0]
    return f"Predicted home price: ${prediction}"

In [ ]:
make_prediction(110, -34.60, -58.46, "Villa Crespo")

#### how predicted home price changes.

In [ ]:
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact


In [ ]:
interact(
    make_prediction,
    area=IntSlider(
        min=max(1, int(X_train["surface_covered"].min())),
        max=max(2, int(X_train["surface_covered"].max())),
        value=int(X_train["surface_covered"].median()),
    ),
    lat=FloatSlider(
        min=float(X_train["lat"].min()),
        max=float(X_train["lat"].max()),
        step=0.01,
        value=float(X_train["lat"].median()),
    ),
    lon=FloatSlider(
        min=float(X_train["lon"].min()),
        max=float(X_train["lon"].max()),
        step=0.01,
        value=float(X_train["lon"].median()),
    ),
    neighborhood=Dropdown(
        options=sorted(X_train["neighborhood"].dropna().unique()),
        value=sorted(X_train["neighborhood"].dropna().unique())[0],
    ),
);


## Author

<a href="https://www.linkedin.com/in/andrew-kalumba-harris/">ANDREW KALUMBA YIGGA</a><br>
<a href =""> </a>


| Date (YYYY-MM-DD) | Prepared By     | 
| ----------------- | --------------  | 
| 2026-08-21        | Author          | 


## <h3 align="center">  Data Science 2026. <h3/>

In [ ]:
model_data=df_clean[df_clean["price_usd"].notna() & (df_clean["price_usd"]>0)].copy()
property_features=[x for x in ["surface_total","surface_covered","rooms","bedrooms","bathrooms","property_type","operation_type"] if x in model_data]
location_features=[x for x in ["lat","lon","neighborhood"] if x in model_data]
combined_features=property_features+location_features
y=np.log1p(model_data["price_usd"])
X_property=model_data[property_features]; X_location=model_data[location_features]; X_combined=model_data[combined_features]
Xp_tr,Xp_te,y_tr,y_te=train_test_split(X_property,y,test_size=.20,random_state=RANDOM_STATE)
Xl_tr,Xl_te,_,_=train_test_split(X_location,y,test_size=.20,random_state=RANDOM_STATE)
Xc_tr,Xc_te,_,_=train_test_split(X_combined,y,test_size=.20,random_state=RANDOM_STATE)
print(len(Xp_tr), len(Xp_te))

In [ ]:
def preprocessor(X):
    nums=X.select_dtypes(include=np.number).columns.tolist()
    cats=X.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer([
        ("num",SimpleImputer(strategy="median",add_indicator=True),nums),
        ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=True))]),cats)])

def ridge_eval(Xtr, Xte, ytr, yte):
    m=Pipeline([("prep",preprocessor(Xtr)),("ridge",Ridge(alpha=1.0))]); m.fit(Xtr, ytr)
    pred=np.expm1(m.predict(Xte)); actual=np.expm1(yte)
    return m,{"MAE":mean_absolute_error(actual,pred),"RMSE":mean_squared_error(actual,pred)**.5,"R2":r2_score(actual,pred)}

ridge_p,met_p=ridge_eval(Xp_tr,Xp_te,y_tr,y_te)
ridge_l,met_l=ridge_eval(Xl_tr,Xl_te,y_tr,y_te)
ridge_c,met_c=ridge_eval(Xc_tr,Xc_te,y_tr,y_te)
print("Property-only",met_p); print("Location-only",met_l); print("Combined",met_c)

## 15. Random Forest 

In [ ]:
rf=Pipeline([("prep",preprocessor(Xc_tr)),("rf",RandomForestRegressor(n_estimators=250,min_samples_leaf=3,random_state=RANDOM_STATE,n_jobs=-1))])
rf.fit(Xc_tr,y_tr)
rf_pred=np.expm1(rf.predict(Xc_te)); actual=np.expm1(y_te)
met_rf={"MAE":mean_absolute_error(actual,rf_pred),"RMSE":mean_squared_error(actual,rf_pred)**.5,"R2":r2_score(actual,rf_pred)}
print(met_rf)

## 16. Model comparison

Lower MAE/RMSE and higher R² are better. Compare property-only, location-only and combined models to answer whether location is more informative than property characteristics.

In [ ]:
results=pd.DataFrame([
 {"Model":"Ridge","Feature set":"Property",**met_p},
 {"Model":"Ridge","Feature set":"Location",**met_l},
 {"Model":"Ridge","Feature set":"Combined",**met_c},
 {"Model":"Random Forest","Feature set":"Combined",**met_rf}])
display(results.sort_values("MAE"))

## 17. Cross-validation

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
cv_model=Pipeline([("prep",preprocessor(X_combined)),("ridge",Ridge(alpha=1.0))])
cv_scores=-cross_val_score(cv_model,X_combined,y,cv=cv,scoring="neg_mean_absolute_error")
print("5-fold CV MAE on log-price:",cv_scores)
print("Mean:",cv_scores.mean(),"Std:",cv_scores.std())

## 18. High-Value classification

The top 10% price threshold becomes a binary target. Logistic regression provides an interpretable baseline for identifying characteristics associated with high-value properties.

In [ ]:
cls=model_data.copy(); cls["high_value"]=(cls["price_usd"]>=cls["price_usd"].quantile(.90)).astype(int)
X=cls[combined_features]; y_cls=cls["high_value"]
Xtr,Xte,ytrc,ytec=train_test_split(X,y_cls,test_size=.20,stratify=y_cls,random_state=RANDOM_STATE)
logit=Pipeline([("prep",preprocessor(Xtr)),("logit",LogisticRegression(max_iter=1000,class_weight="balanced"))])
logit.fit(Xtr,ytrc); pred=logit.predict(Xte); prob=logit.predict_proba(Xte)[:,1]
print("Accuracy:",accuracy_score(ytec,pred)); print("ROC-AUC:",roc_auc_score(ytec,prob)); print(classification_report(ytec,pred))